# Colab Optimized Fine-Tuning with Auto-Resume

This notebook is specifically designed for Google Colab to handle the longest 100,000 sentences of the WMT19 dataset.

**Key Features:**
1. **Google Drive Integration**: Saves checkpoints directly to your Google Drive so you don't lose them when Colab disconnects.
2. **Hugging Face Hub Sync**: Pushes your checkpoints to the Hugging Face Hub during training.
3. **Auto-Resume**: If Colab disconnects, simply run all cells again. It will detect the latest checkpoint in your Google Drive and resume training exactly where it left off.

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub

In [ ]:
import os

# 1. Setup save directory for Kaggle output
# Kaggle does not use Google Drive. Files saved strictly to /kaggle/working/
output_dir = "/kaggle/working/wmt-zh-en-lora"
print(f"✅ Kaggle environment detected. Checkpoints will be saved to: {output_dir}")

# Ensure directory exists
os.makedirs(output_dir, exist_ok=True)

In [ ]:
from huggingface_hub import login

# 2. Login to Hugging Face
hf_token = input("Enter your Hugging Face WRITE Token: ")
login(token=hf_token)

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

# --- CONFIGURATION ---
model_id = "CohereForAI/aya-expanse-8b"
hub_model_id = "YOUR_HF_USERNAME/aya-expanse-8b-wmt-zh-en" # ⚠️ CHANGE THIS to your actual HF username!

In [ ]:
print("Loading multilingual parallel datasets...")
from datasets import load_dataset, concatenate_datasets
import random

print("Fetching English to Egyptian Arabic (arz) data...")
try:
    ds_arz = load_dataset("Helsinki-NLP/tatoeba", lang1="arz", lang2="en", split="train")
except:
    ds_arz = load_dataset("Helsinki-NLP/opus-100", "ar-en", split="train")

print("Fetching Czech to German (cs-de) data...")
ds_cs_de = load_dataset("Helsinki-NLP/tatoeba", lang1="cs", lang2="de", split="train")

def format_arz(example):
    en_val = example['translation']['en']
    arz_val = example['translation'].get('arz', example['translation'].get('ar', ''))
    # Use string concatenation to avoid python splitting bugs during code generation
    return {"text": "Translate from English to Egyptian Arabic:\nEnglish: " + str(en_val) + "\nArabic: " + str(arz_val)}

def format_cs(example):
    cs_val = example['translation']['cs']
    de_val = example['translation']['de']
    return {"text": "Translate from Czech to German:\nCzech: " + str(cs_val) + "\nGerman: " + str(de_val)}

print("Formatting prompts...")
ds_arz_fmt = ds_arz.map(format_arz, remove_columns=ds_arz.column_names)
ds_cs_fmt = ds_cs_de.map(format_cs, remove_columns=ds_cs_de.column_names)

# Take exactly 30,000 sentences from each language pair
arz_target = min(30000, len(ds_arz_fmt))
cs_de_target = min(30000, len(ds_cs_fmt))

print(f"Selecting {arz_target} English-Arabic samples and {cs_de_target} Czech-German samples...")
ds_arz_sampled = ds_arz_fmt.shuffle(seed=42).select(range(arz_target))
ds_cs_sampled = ds_cs_fmt.shuffle(seed=42).select(range(cs_de_target))

print("Combining and perfectly balancing the dataset...")
combined_dataset = concatenate_datasets([ds_arz_sampled, ds_cs_sampled]).shuffle(seed=42)

split_dataset = combined_dataset.train_test_split(test_size=1000, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Dataset ready! Final Train Size: {len(train_dataset)} | Eval Size: {len(eval_dataset)}")


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

previous_lora_path = "/kaggle/working/wmt-zh-en-lora" 

if os.path.exists(previous_lora_path):
    print(f"Loading PREVIOUS LoRA weights from {previous_lora_path} ...")
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, previous_lora_path, is_trainable=True)
else:
    print("Previous weights not found! Falling back to fresh LoRA adapter.")
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
\n

In [ ]:
from trl import SFTConfig, SFTTrainer
import torch

torch.cuda.empty_cache()

sft_config = SFTConfig(
    output_dir="/kaggle/working/aya-continued-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}, 
    
    optim="paged_adamw_8bit",
    
    num_train_epochs=10,             # CHANGED: 10 epochs
    eval_strategy="epoch",           # CHANGED: Evaluate at epoch end
    save_strategy="epoch",           # CHANGED: Save at epoch end
    logging_strategy="epoch",        # CHANGED: Display loss at epoch end
    load_best_model_at_end=True,     # CHANGED: Keep best model based on loss!
    
    learning_rate=2e-4,
    fp16=False,                 
    warmup_ratio=0.03,            
    group_by_length=True,
    lr_scheduler_type="cosine",
    push_to_hub=False,          
    report_to="none",
    
    dataset_text_field="text",
    packing=False,
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset, 
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting continued training...")
trainer.train()

trainer.model.save_pretrained("/kaggle/working/aya-continued-lora-final")
print(f"Continued Training complete! Best Model saved.")
\n

In [ ]:
# unbabel-comet is horribly incompatible with the newest transformers library (4.49+)! 
# We MUST violently downgrade transformers to an older stable version (4.44.2) for comet to import!
# Since we already finished training the model, it is perfectly safe to downgrade.
!pip install -q evaluate unbabel-comet pytorch-lightning "transformers<4.45"


In [ ]:
import torch
import tarfile
import urllib.request
import tempfile
import os
import json
from tqdm.auto import tqdm

print("Downloading FLORES-200 Evaluation Dataset directly from Meta...")
# Because 'trust_remote_code' and 'datasets' loader scripts are now largely deprecated in newer
# HF environments, we bypass the library entirely and pull the pure FLORES-200 tar.gz directly.

samples_to_eval = 200
sources = []
references = []

# Download and extract the pure text sentences in memory safely
with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz", tmp.name)
    with tarfile.open(tmp.name, "r:gz") as tar:
        eng_file = tar.extractfile("./flores200_dataset/dev/eng_Latn.dev")
        zho_file = tar.extractfile("./flores200_dataset/dev/zho_Hans.dev")
        
        # Read exactly the first 200 samples
        sources = [line.decode("utf-8").strip() for line in eng_file.readlines()][:samples_to_eval]
        references = [line.decode("utf-8").strip() for line in zho_file.readlines()][:samples_to_eval]

# Clean up
os.remove(tmp.name)

predictions = []

print(f"Generating translations for {samples_to_eval} samples using the trained LoRA model...")
model.eval()

# We already translated it on your system previously! To avoid waiting another 12 minutes,
# we will just recalculate the predictions if they were lost during an error!
for i, src_text in enumerate(tqdm(sources, desc="Translating")):
    prompt_eval = f"Translate from English to Simplified Chinese:\nEnglish: {src_text}\nChinese:"
    inputs_eval = tokenizer(prompt_eval, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs_eval = model.generate(
            **inputs_eval, 
            max_new_tokens=150, 
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False # Greedy decoding is preferred for precise translation benchmarking
        )
        
    num_gen = outputs_eval.shape[1] - inputs_eval.input_ids.shape[1]
    pred = tokenizer.decode(outputs_eval[0][-num_gen:], skip_special_tokens=True).strip().replace('\n', ' ')
    predictions.append(pred)

print("\nCreating pure Python script to bypass CLI errors...")

# 🚀 INCREDIBLE BYPASS: The CLI crashed because of an argparse python core bug on Kaggle.
# So we will write a literal pure python script to disk that has a completely isolated memory space,
# run it as a standalone app, and print out the result safely.

data = [
    {"src": src, "mt": mt, "ref": ref}
    for src, mt, ref in zip(sources, predictions, references)
]
with open("data_to_grade.json", "w", encoding="utf-8") as f:
    json.dump(data, f)

isolated_script = """
import json
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

from comet import download_model, load_from_checkpoint

print("Downloading COMET model quietly...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

with open("data_to_grade.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Scoring translations...")
comet_results = comet_model.predict(data, batch_size=8, gpus=1)

print("="*40)
print(f"🌟 Final COMET Score (200 Samples): {comet_results.system_score:.4f}")
print("="*40)
"""

with open("run_comet.py", "w", encoding="utf-8") as f:
    f.write(isolated_script)
    
!python run_comet.py

In [ ]:
import shutil
from IPython.display import FileLink
import os

folder_path = '/kaggle/working/wmt-zh-en-lora'

# Calculate the precise size before zipping
total_size = 0
for dirpath, dirnames, filenames in os.walk(folder_path):
    for f in filenames:
        fp = os.path.join(dirpath, f)
        total_size += os.path.getsize(fp)

print(f"The LoRA model raw folder size is: {total_size / (1024 * 1024):.2f} MB")
print("Zipping the model folder for you to download...")

# Zip the entire model folder into one file
shutil.make_archive('/kaggle/working/wmt-zh-en-lora-saved', 'zip', folder_path)

# Create a clickable download link in Kaggle
print("\n✅ Click the link below to download your fine-tuned model directly to your laptop!")
display(FileLink(r'wmt-zh-en-lora-saved.zip'))